In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-addon-aqc-tensor qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# البدء مع التجميع الكمي التقريبي باستخدام شبكات الموترات (AQC-Tensor)


<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
qiskit-addon-utils~=0.3.0
qiskit-addon-aqc-tensor[aer,quimb-jax]~=0.2.0; sys.platform != 'darwin'
scipy~=1.16.3
```
</details>

يوضّح هذا الدليل مثالاً عملياً بسيطاً للبدء مع AQC-Tensor. في هذا المثال، ستأخذ دائرة تروتر التي تحاكي تطور نموذج إيزينغ ذي المجال المستعرض، وتستخدم طريقة AQC-Tensor لتقليل عمق الدائرة الناتجة. بالإضافة إلى ذلك، يتطلب هذا المثال حزمة `qiskit-addon-utils` لتوليد المسألة، و`qiskit-aer` لمحاكاة شبكة الموترات، و`scipy` لتحسين المعاملات.

للبدء، تذكّر أن هاميلتوني نموذج إيزينغ ذي المجال المستعرض يأخذ الشكل

$$ \mathcal{H}_{Ising} = \sum_{i=1}^N J_{i,(i+1)}Z_iZ_{i+1} + h_i X_i $$

حيث نفترض شروط حدودية دورية تعني أنه عند $i=10$ نحصل على $i+1=11\rightarrow 1$، و$J$ هي قوة الاقتران بين موقعين، و$h$ هي شدة المجال المغناطيسي الخارجي.

يولّد مقطع الكود التالي هاميلتوني سلسلة إيزينغ المكوّنة من 10 مواقع مع شروط حدودية دورية.

In [1]:
from qiskit.transpiler import CouplingMap
from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit.synthesis import SuzukiTrotter
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_aqc_tensor import generate_ansatz_from_circuit
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_aer import AerSimulator
from scipy.optimize import OptimizeResult, minimize


# Generate some coupling map to use for this example
coupling_map = CouplingMap.from_heavy_hex(3, bidirectional=False)

# Choose a 10-qubit circle on this coupling map
reduced_coupling_map = coupling_map.reduce(
    [0, 13, 1, 14, 10, 16, 4, 15, 3, 9]
)

# Get a qubit operator describing the Ising field model
hamiltonian = generate_xyz_hamiltonian(
    reduced_coupling_map,
    coupling_constants=(0.0, 0.0, 1.0),
    ext_magnetic_field=(0.4, 0.0, 0.0),
)

## تقسيم تطور الزمن إلى جزأين للتنفيذ الكلاسيكي والكمي
الهدف العام من هذا المثال هو محاكاة التطور الزمني لهاميلتوني النموذج. نقوم بذلك هنا عبر تطور تروتر، الذي سيُقسَّم إلى جزأين.

1. جزء أولي قابل للمحاكاة باستخدام حالات حاصل ضرب المصفوفات (MPS). هذا هو الجزء الذي سيُجمَّع باستخدام AQC-Tensor.
2. جزء لاحق سيُنفَّذ على أجهزة الكم.

سنختار تطوير النظام حتى الزمن $t_f=5$ واستخدام AQC-Tensor لضغط التطور الزمني حتى الزمن $t=4$، ثم نتابع التطور باستخدام خطوات تروتر العادية حتى $t_f$.

من هنا، سنولّد دائرتين: الأولى ستُضغط باستخدام AQC-Tensor والثانية ستُنفَّذ على وحدة معالجة كمومية (QPU). بالنسبة للدائرة الأولى، وبما أنها ستُحاكى كلاسيكياً باستخدام حالات حاصل ضرب المصفوفات، سنستخدم عدداً سخياً من الطبقات لتقليل خطأ تروتر. أما الدائرة الثانية التي تحاكي التطور الزمني من $t_i=4$ إلى $t_f=5$ فستستخدم طبقات أقل بكثير لتقليل العمق.

In [2]:
# Generate circuit to be compressed
aqc_evolution_time = 4.0
aqc_target_num_trotter_steps = 45

aqc_target_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
    time=aqc_evolution_time,
)

# Generate circuit to execute on hardware
subsequent_evolution_time = 1.0
subsequent_num_trotter_steps = 5

subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=subsequent_num_trotter_steps),
    time=subsequent_evolution_time,
)

لأغراض المقارنة، سنولّد أيضاً دائرة ثالثة؛ وهي دائرة تتطور حتى $t=4$، لكن بنفس عدد طبقات الدائرة الثانية التي تتطور من $t_i=4$ إلى $t_f=5$. هذه هي الدائرة التي *كنّا سنستخدمها* لو لم نلجأ إلى تقنية AQC-Tensor. سنسميها دائرة *المقارنة* في الوقت الحالي.

In [3]:
aqc_comparison_num_trotter_steps = int(
    subsequent_num_trotter_steps
    / subsequent_evolution_time
    * aqc_evolution_time
)
aqc_comparison_num_trotter_steps

comparison_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_comparison_num_trotter_steps),
    time=aqc_evolution_time,
)

## توليد الأنساتز وبناء محاكاة MPS
بعد ذلك سنولّد الأنساتز الذي سنحسّنه. سيتطور حتى نفس زمن التطور للدائرة الأولى (من $t_i=0$ إلى $t_f=4$)، لكن بعدد أقل من خطوات تروتر.

بمجرد توليد الدائرة، نمرّرها إلى دالة `generate_ansatz_from_circuit()` في AQC-Tensor، التي تحلّل ترابط الكيوبتات ثنائية وتُرجع شيئين: أولاً دائرة أنساتز مولَّدة بنفس ترابط الكيوبتات ثنائية، وثانياً مجموعة من المعاملات التي عند إدخالها في الأنساتز تُنتج الدائرة المُدخلة.

In [4]:
aqc_ansatz_num_trotter_steps = 5

aqc_good_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
    time=aqc_evolution_time,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit, qubits_initially_zero=True
)
aqc_ansatz.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg)

بعد ذلك سنبني تمثيل MPS للحالة التي سيقرّبها AQC. كما سنحسب الإخلاص $|\langle\psi_1 | \psi_2 \rangle |^2$ بين الحالة التي تُعدّها دائرة المقارنة والدائرة التي تولّد الحالة المستهدفة (التي استخدمت خطوات تروتر أكثر).

In [5]:
# Generate MPS simulator settings and obtain the MPS representation of the target state
simulator_settings = AerSimulator(
    method="matrix_product_state",
    matrix_product_state_max_bond_dimension=100,
)
aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)


# Compute the fidelity between the MPS representation of the target state and the state produced by the comparison circuit
comparison_mps = tensornetwork_from_circuit(
    comparison_circuit, simulator_settings
)
comparison_fidelity = (
    abs(compute_overlap(comparison_mps, aqc_target_mps)) ** 2
)
print(f"Comparison fidelity: {comparison_fidelity}")

Comparison fidelity: 0.9997111919739693


## Optimize the parameters of the ansatz using the MPS

Lastly, we will optimize the ansatz circuit such that it produces the target state better than our `comparison_fidelity`. The cost function to minimize will be the `MaximizeStateFidelity` and will be optimized using scipy's L-BFGS optimizer.

In [6]:
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)

stopping_point = 1 - comparison_fidelity


def callback(intermediate_result: OptimizeResult):
    print(f"Intermediate result: Fidelity {1 - intermediate_result.fun:.8}")
    if intermediate_result.fun < stopping_point:
        # Good enough for now
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 100},
    callback=callback,
)
if (
    result.status
    not in (
        0,
        1,
        99,
    )
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

Intermediate result: Fidelity 0.95084365


Intermediate result: Fidelity 0.98409893


Intermediate result: Fidelity 0.99142033


Intermediate result: Fidelity 0.99521405


Intermediate result: Fidelity 0.99566673


...

Intermediate result: Fidelity 0.99956469


Intermediate result: Fidelity 0.99964488


Intermediate result: Fidelity 0.99967419


Intermediate result: Fidelity 0.99968821


Intermediate result: Fidelity 0.9997448
Done after 24 iterations.


## تحسين معاملات الأنساتز باستخدام MPS
أخيراً، سنحسّن دائرة الأنساتز بحيث تُنتج الحالة المستهدفة بإخلاص أعلى مما ستُنتجه `comparison_fidelity`. دالة التكلفة التي سنقلّلها هي `MaximizeStateFidelity` وسيتم تحسينها باستخدام محسّن L-BFGS من scipy.

In [7]:
final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)
final_circuit.compose(subsequent_circuit, inplace=True)
final_circuit.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/45abbabe-0289-4a09-aa99-89f70bdc535d-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution) tutorial.
</Admonition>